# New Section

In [2]:
# ===== CELL 1: load =====
import pandas as pd
import plotly.express as px

df = pd.read_csv("orders_clean.csv", parse_dates=["order_date", "payment_date", "due_date", "return_date"])
biz = df[df["segment"] != "Consumer"]          # business customers only (they have a rep)
print(len(df), "orders,", f"${df['revenue'].sum():,.0f} revenue")


6136 orders, $7,973,577 revenue


In [3]:
# ===== CELL 2: INSIGHT 1a - revenue grows faster than profit =====
jan_sep = df[df["order_date"].dt.month <= 9]    # same 9 months in both years
yoy = jan_sep.groupby(jan_sep["order_date"].dt.year)[["revenue", "margin"]].sum()
yoy["margin_pct"] = yoy["margin"] / yoy["revenue"]
print(yoy.round(3))
print("Revenue growth:", f"{yoy.loc[2026,'revenue']/yoy.loc[2025,'revenue']-1:.0%}",
      "| Margin growth:", f"{yoy.loc[2026,'margin']/yoy.loc[2025,'margin']-1:.0%}")

                revenue       margin  margin_pct
order_date                                      
2025        2930934.358   875691.113       0.299
2026        3898558.439  1061595.649       0.272
Revenue growth: 33% | Margin growth: 21%


In [ ]:
# ===== CELL 3: INSIGHT 1b - one rep discounts far more than the others =====
reps = biz.groupby("sales_rep").agg(
    region=("region", "first"),
    revenue=("revenue", "sum"),
    margin=("margin", "sum"),
    avg_discount=("discount", "mean"),
    loss_orders=("margin", lambda m: (m < 0).sum()),
)
reps["margin_pct"] = reps["margin"] / reps["revenue"]
print(reps.round(3))

fig = px.scatter(reps.reset_index(), x="avg_discount", y="margin_pct", size="revenue",
                 text="sales_rep", color="region",
                 title="R07 (Atlantic) sells the most, at double the discount and half the margin",
                 labels={"avg_discount": "Average discount", "margin_pct": "Margin %"})
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_tickformat=".0%", yaxis_tickformat=".0%")
fig.show()

             region      revenue      margin  avg_discount  loss_orders  \
sales_rep                                                                 
R01          Quebec   556717.810  183372.395         0.091            0   
R02          Quebec   266971.466   90753.411         0.077            0   
R03          Quebec   620995.027  199202.422         0.096            0   
R04         Ontario   740713.158  238734.398         0.090            0   
R05         Ontario   740244.902  236854.762         0.095            0   
R06         Ontario  1074824.539  334092.464         0.101            0   
R07        Atlantic  1976827.839  315505.144         0.232           40   
R08        Prairies   440174.180  137945.160         0.092            0   
R09        Prairies   356716.310  113668.375         0.114            0   
R10              BC   406406.924  131472.074         0.090            0   
R11              BC   387851.131  126514.721         0.086            0   

           margin_pct  


In [ ]:
# ===== CELL 4: INSIGHT 1c - quarter-end loading =====
last7 = biz[biz["days_to_quarter_end"] < 7]
share = (last7.groupby("sales_rep")["revenue"].sum() / biz.groupby("sales_rep")["revenue"].sum())
print("Share of revenue booked in the last 7 days of a quarter:\n", share.round(3))

res = biz[biz["segment"] == "Reseller"]
share_res = (res[res["days_to_quarter_end"] < 7].groupby("sales_rep")["revenue"].sum()
             / res.groupby("sales_rep")["revenue"].sum())
print("Same, resellers only:\n", share_res.round(3))

fig = px.bar(share.reset_index(name="share"), x="sales_rep", y="share",
             title="44% of R07's revenue lands in the last week of the quarter (others: 3-9%)",
             labels={"share": "Share of revenue in last 7 days", "sales_rep": "Sales rep"})
fig.update_layout(yaxis_tickformat=".0%")
fig.show()


Share of revenue booked in the last 7 days of a quarter:
 sales_rep
R01    0.094
R02    0.069
R03    0.064
R04    0.078
R05    0.093
R06    0.072
R07    0.438
R08    0.070
R09    0.065
R10    0.061
R11    0.032
Name: revenue, dtype: float64
Same, resellers only:
 sales_rep
R01    0.094
R03    0.137
R04    0.062
R05    0.100
R06    0.055
R07    0.704
R08    0.141
R09    0.011
R10    0.081
Name: revenue, dtype: float64


In [ ]:
# ===== CELL 5: INSIGHT 1d - and those orders come back =====
over = df[df["return_reason"] == "Overstock / ordered in error"]
print("Overstock returns:", len(over), f"| refunds ${over['refund_amount'].sum():,.0f}")
print(over.groupby(["sales_rep", "segment", "region"]).size())
print("Days before quarter end when these orders were placed:", sorted(over["days_to_quarter_end"].unique()))

# Atlantic: revenue up, margin down
atl = jan_sep.pivot_table(index="region", columns=jan_sep["order_date"].dt.year,
                          values=["revenue", "margin"], aggfunc="sum").round(0)
print(atl)


Overstock returns: 46 | refunds $150,538
sales_rep  segment   region  
R07        Reseller  Atlantic    46
dtype: int64
Days before quarter end when these orders were placed: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
              margin              revenue           
order_date      2025      2026       2025       2026
region                                              
Atlantic    156985.0  115512.0   760922.0   954649.0
BC           98082.0  142900.0   293951.0   441768.0
Ontario     336245.0  407795.0  1024068.0  1285079.0
Prairies     97062.0  133457.0   303578.0   417918.0
Quebec      187317.0  261931.0   548415.0   799145.0


In [ ]:
# ===== CELL 6: INSIGHT 2a - damage is a desk problem =====
dmg = df[df["return_reason"] == "Damaged in transit"]
print(dmg.groupby("product").agg(returns=("order_id", "count"), refunds=("refund_amount", "sum"))
        .sort_values("refunds", ascending=False).round(0))


                        returns   refunds
product                                  
ErgoLift Standing Desk       79  248260.0
Office Chair                  6   12119.0
Webcam HD                     3    3173.0
Monitor 27"                   3    2165.0
Monitor 24"                   2    1759.0
Laptop Stand                  3     515.0
Wireless Mouse                4     323.0
Desk Lamp                     2     271.0
USB-C Dock                    1     140.0


In [ ]:
# ===== CELL 7: INSIGHT 2b - ...in two regions =====
desks = df[df["product"] == "ErgoLift Standing Desk"].copy()
desks["damaged"] = desks["return_reason"] == "Damaged in transit"
by_region = desks.groupby("region").agg(orders=("order_id", "count"), damaged=("damaged", "sum"),
                                        margin=("margin", "sum"))
by_region["damage_rate"] = by_region["damaged"] / by_region["orders"]
by_region["damage_refunds"] = desks[desks["damaged"]].groupby("region")["refund_amount"].sum()
print(by_region.round(3))

fig = px.bar(by_region.reset_index().sort_values("damage_rate"), x="damage_rate", y="region",
             orientation="h", title="About 30% of desks shipped to BC and the Prairies arrive damaged",
             labels={"damage_rate": "Desk orders damaged in transit", "region": ""})
fig.add_vline(x=0.10, line_dash="dash", annotation_text="10% alert threshold")
fig.update_layout(xaxis_tickformat=".0%")
fig.show()

          orders  damaged      margin  damage_rate  damage_refunds
region                                                            
Atlantic     112        5   78758.713        0.045        15412.90
BC            71       22   71556.741        0.310        74613.53
Ontario      244       17  257059.558        0.070        49050.29
Prairies      74       22   84366.015        0.297        64483.42
Quebec       153       13  143765.981        0.085        44700.14


In [ ]:
# ===== CELL 8: INSIGHT 2c - damaged deliveries and quiet customers =====
LAST_DAY = df["order_date"].max()
cust = biz.groupby("customer_id").agg(last_order=("order_date", "max"), orders=("order_id", "count"),
                                      segment=("segment", "first"), revenue=("revenue", "sum"))
cust["days_quiet"] = (LAST_DAY - cust["last_order"]).dt.days
cust["quiet"] = cust["days_quiet"] > 90
cust["had_damage"] = cust.index.isin(dmg["customer_id"])
regulars = cust[cust["orders"] >= 5]            # ignore customers who barely ordered
link = regulars.groupby("had_damage")["quiet"].agg(share_quiet="mean", quiet="sum", customers="count")
print(link.round(3))

fig = px.bar(link.reset_index().replace({True: "Had a damaged delivery", False: "No damaged delivery"}),
             x="had_damage", y="share_quiet",
             title="Customers who got a damaged delivery are 3x more likely to stop ordering",
             labels={"had_damage": "", "share_quiet": "Share with no order in 90+ days"})
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

            share_quiet  quiet  customers
had_damage                               
False             0.102     22        215
True              0.338     22         65


In [4]:
# ===== CELL 9: CHECK - unpaid invoices are a small problem =====
unpaid = biz[biz["is_unpaid"]]
print("Unpaid invoices:", len(unpaid), f"${unpaid['revenue'].sum():,.0f}")
for days in [0, 30, 60]:
    late = unpaid[unpaid["days_overdue"] > days]
    print(f"  more than {days} days overdue: {len(late)} invoices, ${late['revenue'].sum():,.0f}")
print("Average days to pay, by quarter:\n", biz.groupby("quarter")["days_to_pay"].mean().round(1))

Unpaid invoices: 92 $250,004
  more than 0 days overdue: 54 invoices, $135,367
  more than 30 days overdue: 20 invoices, $52,939
  more than 60 days overdue: 9 invoices, $28,105
Average days to pay, by quarter:
 quarter
2025Q1    39.3
2025Q2    40.4
2025Q3    40.6
2025Q4    39.5
2026Q1    40.7
2026Q2    39.2
2026Q3    33.9
Name: days_to_pay, dtype: float64


In [ ]:
# ===== CELL 10: DETECTION - damage rate per product and region vs a threshold =====
import pandas as pd
import plotly.express as px

df = pd.read_csv("orders_clean.csv", parse_dates=["order_date", "payment_date", "due_date", "return_date"])

DAMAGE_THRESHOLD = 0.10   # alert when more than 10% of orders arrive damaged
MIN_ORDERS = 20           # ignore product/region pairs with too few orders to judge

df["damaged"] = df["return_reason"] == "Damaged in transit"
damage = df.groupby(["product", "region"]).agg(orders=("order_id", "count"),
                                               damaged=("damaged", "sum"),
                                               refunds=("refund_amount", lambda s: s[df.loc[s.index, "damaged"]].sum()))
damage["damage_rate"] = damage["damaged"] / damage["orders"]
damage["alert"] = (damage["damage_rate"] > DAMAGE_THRESHOLD) & (damage["orders"] >= MIN_ORDERS)

damage_alerts = damage[damage["alert"]].sort_values("damage_rate", ascending=False)
print(f"{len(damage_alerts)} product/region pairs above {DAMAGE_THRESHOLD:.0%}:")
print(damage_alerts.round(3))

grid = damage["damage_rate"].unstack("region")
fig = px.imshow(grid, text_auto=".0%", aspect="auto", color_continuous_scale="Reds",
                zmin=0, zmax=max(0.30, DAMAGE_THRESHOLD),
                title=f"Damaged-in-transit rate by product and region (alert above {DAMAGE_THRESHOLD:.0%})",
                labels={"x": "Region", "y": "", "color": "Damage rate"})
fig.show()

2 product/region pairs above 10%:
                                 orders  damaged   refunds  damage_rate  alert
product                region                                                 
ErgoLift Standing Desk BC            71       22  74613.53        0.310   True
                       Prairies      74       22  64483.42        0.297   True


In [ ]:
# ===== CELL 11: DETECTION - customers close to their credit limit =====
AS_OF = pd.Timestamp("2026-10-31")   # "today". Try 2026-09-30 to see another day.
WARN, URGENT = 0.75, 0.90

biz = df[df["segment"] != "Consumer"]
# An invoice is open on AS_OF if it was ordered by then and not yet paid by then.
open_inv = biz[(biz["order_date"] <= AS_OF) & (biz["payment_date"].isna() | (biz["payment_date"] > AS_OF))].copy()

credit = open_inv.groupby("customer_id").agg(owed=("revenue", "sum"), open_invoices=("order_id", "count"),
                                             credit_limit=("credit_limit", "first"), segment=("segment", "first"),
                                             payment_terms=("payment_terms", "first"), rep=("account_rep", "first"))
credit["used"] = credit["owed"] / credit["credit_limit"]
credit["level"] = pd.cut(credit["used"], [0, WARN, URGENT, 1, float("inf")],
                         labels=["OK", "Warning (75%+)", "Urgent (90%+)", "Over limit"], right=False)
credit_alerts = credit[credit["used"] >= WARN].sort_values("used", ascending=False)
print(f"As of {AS_OF.date()}: {len(credit)} customers owe money, {len(credit_alerts)} at {WARN:.0%}+ of their limit")
print(credit_alerts.round(2))

As of 2026-10-31: 61 customers owe money, 1 at 75%+ of their limit
                 owed  open_invoices  credit_limit     segment payment_terms  \
customer_id                                                                    
C1173        27578.97              9         25000  Enterprise        Net 60   

             rep  used       level  
customer_id                         
C1173        R04   1.1  Over limit  


In [ ]:
# ===== CELL 12: DETECTION - invoices due within 5 days, and invoices already overdue =====
DUE_SOON_DAYS = 5

open_inv["days_until_due"] = (open_inv["due_date"] - AS_OF).dt.days
open_inv["status"] = "Not due yet"
open_inv.loc[open_inv["days_until_due"].between(0, DUE_SOON_DAYS), "status"] = f"Due within {DUE_SOON_DAYS} days"
open_inv.loc[open_inv["days_until_due"] < 0, "status"] = "Overdue"

cols = ["order_id", "customer_id", "segment", "payment_terms", "account_rep", "due_date", "days_until_due", "revenue"]
due_soon = open_inv[open_inv["status"] == f"Due within {DUE_SOON_DAYS} days"][cols].sort_values("days_until_due")
overdue = open_inv[open_inv["status"] == "Overdue"][cols].sort_values("days_until_due")

print(open_inv.groupby(["payment_terms", "status"]).agg(invoices=("order_id", "count"), amount=("revenue", "sum")).round(0))
print(f"\nDue within {DUE_SOON_DAYS} days -> send a reminder:\n", due_soon.assign(revenue=due_soon["revenue"].round(0)).to_string(index=False))
print(f"\nOverdue: {len(overdue)} invoices, ${overdue['revenue'].sum():,.0f}")

                                 invoices    amount
payment_terms status                               
Net 30        Overdue                  34   71383.0
Net 60        Due within 5 days         2    3440.0
              Not due yet              36  111197.0
              Overdue                  20   63984.0

Due within 5 days -> send a reminder:
  order_id customer_id    segment payment_terms account_rep   due_date  days_until_due  revenue
SO-006369       C1159 Enterprise        Net 60         R03 2026-11-04               4    884.0
SO-006389       C1155 Enterprise        Net 60         R07 2026-11-05               5   2556.0

Overdue: 54 invoices, $135,367
